# Observatorio MinCiencias · Notebook de introduccion

Bienvenido. Este notebook te muestra como cargar y empezar a explorar los datos del **Observatorio de Investigadores Reconocidos de MinCiencias** (2013-2021), construido en Ustadistica 2026-I.

**Objetivo:** en 5-10 minutos sabras que hay en el dataset y como empezar a hacer tus propios analisis.

**Lo que veras:**
1. Como cargar el dataset consolidado de seis convocatorias.
2. Que variables tiene cada investigador.
3. Tu primer grafico: donde se concentran los investigadores reconocidos en Colombia.

**Este notebook es autocontenido:** todo lo que necesitas vive en esta misma carpeta. No necesitas clonar repositorios ni instalar paquetes del proyecto. Solo Python con tres librerias estandar.

## Setup

Necesitas tener instalado Python 3.10 o superior y las siguientes librerias:

```bash
pip install pandas matplotlib pyyaml
```

Y el archivo `investigadores_consolidado.csv` debe estar en la subcarpeta `datos/` (al lado de este notebook). Si no lo tienes, hay instrucciones para descargarlo en el README.md.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

AQUI = Path.cwd()  # carpeta donde vive este notebook
DATA = AQUI / "datos" / "investigadores_consolidado.csv"
CATALOGO = AQUI / "catalogo.yaml"

print("Notebook ubicado en:", AQUI)
print("Dataset esperado en:", DATA.relative_to(AQUI))
if not DATA.exists():
    print()
    print("!! El archivo de datos NO se encuentra. Necesitas:")
    print("   1) Crear la subcarpeta 'datos/' al lado de este notebook")
    print("   2) Colocar dentro: investigadores_consolidado.csv")
    print("   3) El archivo se descarga desde la API de Socrata")
    print("      (ver README.md para la URL)")
else:
    print("OK: dataset disponible")

## Paso 1 - Cargar los datos

El dataset es la descarga completa desde la **API de Socrata** (datos.gov.co, identificador `bqtm-4y2h`):

- **77.237 apariciones** de investigadores reconocidos
- **30.086 investigadores unicos**
- **6 convocatorias**: 640 (2013), 693 (2014), 737 (2015), 781 (2017), 833 (2018) y 894 (2021)
- **30 columnas** documentadas en `catalogo.yaml` (al lado de este notebook)

Cada fila es la aparicion de un investigador en una convocatoria. Un mismo investigador puede aparecer en varias, por eso el conteo unico es menor que el total de filas.

In [ ]:
df = pd.read_csv(DATA, low_memory=False)
df.columns = df.columns.str.upper()  # normalizamos a MAYUSCULAS

print(f"Filas:                 {df.shape[0]:,}")
print(f"Columnas:              {df.shape[1]}")
print(f"Investigadores unicos: {df['ID_PERSONA_PR'].nunique():,}")
df.head(3)

## Paso 2 · Conocer las variables más usadas

Hay 30 columnas. Las más importantes para arrancar:

In [ ]:
columnas_clave = [
    "ID_PERSONA_PR",           # llave única del investigador
    "NME_CONVOCATORIA",        # convocatoria (640, 693, 737, 781, 833, 894)
    "NME_CLASIFICACION_PR",    # categoria: Junior / Asociado / Senior / Emerito
    "NME_GRAN_AREA_PR",        # area OCDE (Ingenieria, Sociales, Medicas, ...)
    "NME_GENERO_PR",           # genero reportado
    "NME_DEPARTAMENTO_RES_PR", # departamento de residencia
]
df[columnas_clave].head(10)

El **catálogo de datos** documenta las 30 columnas. Lo cargamos para saber qué significa cada una sin abrir otro archivo:

In [ ]:
with open(CATALOGO, encoding="utf-8") as f:
    catalogo = yaml.safe_load(f)

print("Secciones del catalogo:")
for k in catalogo.keys():
    print(" -", k)
print()
print("Sugerencia: abre catalogo.yaml en un editor para ver todas las variables documentadas.")

## Paso 3 · Pregunta concreta

**¿Dónde se concentran los investigadores reconocidos en Colombia?**

Contamos investigadores únicos por departamento de residencia (cada persona se cuenta una sola vez, aunque aparezca en varias convocatorias).

In [ ]:
top10 = (
    df.groupby("NME_DEPARTAMENTO_RES_PR")["ID_PERSONA_PR"]
      .nunique()
      .sort_values(ascending=False)
      .head(10)
)
top10

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
top10.sort_values().plot.barh(ax=ax, color="#1e40af")
ax.set_xlabel("Investigadores únicos (2013–2021)")
ax.set_ylabel("")
ax.set_title("Top 10 departamentos por investigadores reconocidos", fontsize=13, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## Lectura del gráfico

Bogotá y Antioquia concentran cerca del **48% del padrón nacional**. Los siguientes ocho departamentos (Valle, Atlántico, Santander, Cundinamarca, Bolívar, Caldas, Norte de Santander, Risaralda) suman otro ~30%. Los 22 departamentos restantes se reparten lo que queda: territorios como Vichada o Vaupés tienen menos de 50 investigadores reconocidos en toda la década.

Esta concentración territorial es uno de los hallazgos centrales del observatorio.

## Para profundizar

Este notebook es solo el punto de entrada. El observatorio completo del que se extrajo este dataset incluye:

- **Cinco ejes de analisis:** Personas, Produccion, Territorios, Campos OCDE y trayectoria longitudinal (Sankeys de transicion de categoria).
- **Cruce con produccion de grupos** (Socrata `33dq-ab5a`, 3.16 M productos academicos): permite responder cuanto produce cada categoria, area OCDE y territorio.
- **Tabla maestra de instituciones**: 211 IES canonicas que cubren el 91% del padron.
- **Modelo dimensional en DuckDB** y **dashboard interactivo en Streamlit** con 7 pestanas filtrables.

Si quieres ver todo el codigo y los hallazgos, el repositorio del observatorio esta en:
**`github.com/ustadistica/Observatorio_Ministerio_de_Ciencias_Grupo8`**

**Ideas para empezar tu propio analisis con este dataset:**

1. Como cambio el porcentaje de mujeres en Ingenieria entre 2013 y 2021?
2. Cual es la categoria (Junior / Asociado / Senior) con mayor crecimiento?
3. Que departamentos tienen menor edad promedio de sus investigadores?
4. Hay correlacion entre tamano del departamento y area OCDE dominante?
5. Cuantos investigadores fueron clasificados por primera vez en 2021?

El dataset tiene 30 columnas. Hay muchisimo mas para preguntarle.